# 04 DeepSeek V3 架构走读：MoE + MLA

## 简介

本笔记本介绍 DeepSeek V3 的两大核心创新：
- **MLA (Multi-head Latent Attention)**: 低秩压缩 KV Cache，大幅降低推理显存
- **MoE (Mixture of Experts)**: 稀疏激活，以较低的推理计算量获得海量参数

我们将逐步走读 MLA 的解耦 RoPE 设计和 MoE 的 Top-K 路由机制。

## 1. 导入

In [ ]:
import torch
from models.deepseek_v3.config import DeepSeekV3Config
from models.deepseek_v3.mla import MultiHeadLatentAttention
from models.deepseek_v3.moe import Router, SharedExpert, RoutedExpert, MoELayer
from models.deepseek_v3.model import DeepSeekV3Block, DeepSeekV3Model, DeepSeekV3ForCausalLM
from models.common.attention import MultiHeadAttention, GroupedQueryAttention

# 小型演示配置
config = DeepSeekV3Config(
    dim=128, n_heads=4, n_layers=2,
    kv_lora_rank=64, qk_rope_head_dim=16,
    n_routed_experts=8, n_shared_experts=1, n_activated_experts=2,
    moe_intermediate_dim=256, max_seq_len=64,
)
print(f"DeepSeek V3 配置:")
print(f"  dim={config.dim}, heads={config.n_heads}, head_dim={config.head_dim}")
print(f"  kv_lora_rank={config.kv_lora_rank}, qk_rope_head_dim={config.qk_rope_head_dim}")
print(f"  experts: {config.n_routed_experts} routed + {config.n_shared_experts} shared")

## 2. MLA (Multi-head Latent Attention) 原理

MLA 的核心思想：
1. **低秩压缩 KV**: 将输入压缩到 kv_lora_rank 维的潜在向量，推理时只存储这个压缩向量
2. **解耦 RoPE**: 将位置编码施加到 K 的一小部分维度，避免位置信息在压缩中丢失

```
标准 MHA KV Cache: 每 token 存 n_heads × head_dim (K) + n_heads × head_dim (V)
MLA KV Cache:     每 token 存 kv_lora_rank (压缩潜在向量) + qk_rope_head_dim (解耦 RoPE)
```

In [ ]:
mla = MultiHeadLatentAttention(config)
x = torch.randn(2, 8, config.dim)

print("=" * 60)
print("MLA 逐步走读")
print("=" * 60)
print(f"输入 x: {list(x.shape)}")

# 步骤 1: Q 投影
q = mla.w_q(x)
q = q.view(2, 8, config.n_heads, config.head_dim).transpose(1, 2)
print(f"\n1. Q 投影后分头: {list(q.shape)}")

# 步骤 2: KV 压缩 (关键!)
kv_a = mla.w_kv_a(x)
kv_latent = kv_a[:, :, :config.kv_lora_rank]
k_rope_raw = kv_a[:, :, config.kv_lora_rank:]
print(f"2. KV 压缩输出: {list(kv_a.shape)}")
print(f"   - 潜在向量 (存储到 Cache): {list(kv_latent.shape)}")
print(f"   - RoPE 部分 (存储到 Cache): {list(k_rope_raw.shape)}")

# 步骤 3: 上投影 K, V
k = mla.w_k_b(kv_latent)
k = k.view(2, 8, config.n_heads, config.head_dim).transpose(1, 2)
v = mla.w_v_b(kv_latent)
v = v.view(2, 8, config.n_heads, config.head_dim).transpose(1, 2)
print(f"\n3. K 上投影: {list(k.shape)}")
print(f"4. V 上投影: {list(v.shape)}")

# 步骤 4: 解耦 RoPE
q_nope = q[:, :, :, :-config.qk_rope_head_dim]
q_rope = q[:, :, :, -config.qk_rope_head_dim:]
print(f"\n5. Q 非 RoPE 部分: {list(q_nope.shape)}")
print(f"6. Q RoPE 部分:    {list(q_rope.shape)}")

q_rope = mla._apply_rope(q_rope)
k_rope = mla._apply_rope(k_rope_raw)
k_rope = k_rope.unsqueeze(1).expand(2, config.n_heads, 8, config.qk_rope_head_dim)
print(f"7. K RoPE (广播到所有头): {list(k_rope.shape)}")

# 步骤 5: 最终输出
out = mla(x, use_causal_mask=True)
print(f"\n最终输出: {list(out.shape)} (与输入一致: {out.shape == x.shape})")

## 3. KV Cache 大小对比: MHA vs GQA vs MLA

In [ ]:
def kv_cache_per_token_bytes(n_heads, n_kv_heads, head_dim, kv_lora_rank, rope_dim, dtype=2):
    """计算每 token 的 KV Cache 字节数 (float16)
    
    MHA/GQA: 存完整 K + V  (各 n_kv_heads * head_dim 维)
    MLA:     存潜在向量 kv_lora_rank + RoPE 键 rope_dim
    """
    mha_bytes = 2 * n_heads * head_dim * dtype
    gqa_bytes = 2 * n_kv_heads * head_dim * dtype
    mla_bytes = (kv_lora_rank + rope_dim) * dtype
    return mha_bytes, gqa_bytes, mla_bytes

head_dim = 128
n_heads = 32
n_kv_heads = 8
kv_lora_rank = 512
rope_dim = 64

mha_b, gqa_b, mla_b = kv_cache_per_token_bytes(
    n_heads, n_kv_heads, head_dim, kv_lora_rank, rope_dim
)
print(f"每 token KV Cache 大小 (float16):")
print(f"  MHA ({n_heads} heads): {mha_b} bytes = {mha_b/1024:.2f} KB")
print(f"  GQA ({n_kv_heads} KV heads): {gqa_b} bytes = {gqa_b/1024:.2f} KB")
print(f"  MLA ({kv_lora_rank}+{rope_dim}):  {mla_b} bytes = {mla_b/1024:.2f} KB")
print(f"\n  MLA 相对 MHA 节省: {(1 - mla_b/mha_b)*100:.1f}%")
print(f"  MLA 相对 GQA 节省: {(1 - mla_b/gqa_b)*100:.1f}%")

## 4. MoE (Mixture of Experts) 原理

MoE 的核心组件:
1. **Router (Top-K 门控)**: 为每个 token 选择得分最高的 K 个专家
2. **Shared Expert**: 所有 token 都通过的共享专家
3. **Routed Experts**: 只有被路由到的 token 才经过的专家

```
输出 = SharedExpert(x) + Σ_{i in TopK} score_i × RoutedExpert_i(x)
```

In [ ]:
# 演示 Router
router = Router(dim=config.dim, n_experts=config.n_routed_experts,
                top_k=config.n_activated_experts)
x = torch.randn(2, 4, config.dim)

scores, indices = router(x)
print(f"Router 输入: {list(x.shape)}")
print(f"Router 得分: {list(scores.shape)} (Top-{config.n_activated_experts} 归一化得分)")
print(f"Router 索引: {list(indices.shape)} (选中的专家编号)")
print(f"\n第一个 token 的路由结果:")
print(f"  得分: {scores[0, 0].detach().numpy().round(3)}")
print(f"  专家: {indices[0, 0].tolist()}")

In [ ]:
# 完整 MoE Layer 前向传播
moe_layer = MoELayer(config)
x = torch.randn(2, 8, config.dim)

print("=" * 60)
print("MoE Layer 前向传播")
print("=" * 60)
out = moe_layer(x)
print(f"输入: {list(x.shape)}")
print(f"输出: {list(out.shape)}")

# 统计专家选择分布
_, indices = moe_layer.router(x)
flat_indices = indices.flatten()

print(f"\n专家选择分布 (共 {flat_indices.numel()} 个 token, 每 token 选 {config.n_activated_experts} 个):")
for e in range(config.n_routed_experts):
    count = (flat_indices == e).sum().item()
    bar = "█" * (count // 2) if count > 0 else ""
    print(f"  Expert {e}: {count:3d} tokens {bar}")

print(f"\n共享专家: 所有 {x.shape[0] * x.shape[1]} 个 token 都经过")

## 5. DeepSeek V3 Full Forward Pass

In [ ]:
# 完整模型
model = DeepSeekV3Model(config)
lm = DeepSeekV3ForCausalLM(config)

tokens = torch.randint(0, config.vocab_size, (2, 16))

h = model(tokens, use_causal_mask=True)
logits = lm(tokens, use_causal_mask=True)

print(f"DeepSeekV3Model:  {list(tokens.shape)} -> {list(h.shape)}")
print(f"DeepSeekV3ForCausalLM: {list(tokens.shape)} -> {list(logits.shape)}")

print(f"\n总参数量: {sum(p.numel() for p in lm.parameters()):,}")
print(f"  但每个 token 只激活 1 共享专家 + {config.n_activated_experts}/{config.n_routed_experts} 路由专家")
print(f"  实际激活参数: 仅约 1/({config.n_routed_experts}) 的总参数")

## 6. MoE 激活参数分析

In [ ]:
# 计算一个路由专家的参数量
expert = moe_layer.routed_experts[0]
expert_params = sum(p.numel() for p in expert.parameters())
shared_params = sum(p.numel() for p in moe_layer.shared_expert.parameters())

total_expert_params = shared_params + config.n_routed_experts * expert_params
active_expert_params = shared_params + config.n_activated_experts * expert_params

print(f"MoE 专家参数量分析:")
print(f"  单个路由专家参数: {expert_params:,}")
print(f"  共享专家参数: {shared_params:,}")
print(f"  所有专家总参数: {total_expert_params:,}")
print(f"  每个 token 激活参数: {active_expert_params:,}")
print(f"  激活率: {active_expert_params/total_expert_params*100:.1f}% "
      f"(激活 {config.n_activated_experts + 1}/{config.n_routed_experts + 1} 个专家)")

print(f"\n★ MoE 的核心优势: 参数总量大 (知识容量大) 但单次激活量小 (推理快)")

## 8. 相关文档

本 notebook 对应的详细原理文档：

- [DeepSeek V3 架构详解](../../docs/models/deepseek-v3.md) — MLA 低秩压缩、MoE Router/SharedExpert/RoutedExpert 的完整原理
- [Attention 机制详解](../../docs/models/attention.md) — 与 MHA/GQA 的对比
- [专家并行详解](../../docs/parallel/expert-parallel.md) — MoE 模型的分布式部署

建议阅读顺序：先运行本 notebook 建立直觉，再阅读文档理解数学细节。

## 8. 练习与思考

1. **MLA vs GQA**: 修改 `kv_lora_rank` 参数，对比 MLA 和 GQA 在不同配置下的 KV Cache 大小
2. **Top-K 调参**: 将 `n_activated_experts` 从 2 改为 1 或 4，观察负载分布变化
3. **共享 Expert 分析**: 去掉 SharedExpert，对比有无共享 Expert 时的模型参数利用率

## 7. 关键要点总结

1. **MLA 低秩压缩**: 将 KV Cache 从 `n_heads × head_dim` 压缩到 `kv_lora_rank`，节省 80-90% KV Cache 内存
2. **解耦 RoPE**: 只对 K 的一小部分维度施加旋转位置编码，避免压缩破坏位置信息
3. **MoE 稀疏激活**: 每个 token 只激活 Top-K 个专家，参数总量大但推理计算量小
4. **路由器负载均衡**: 实际训练中需添加辅助损失防止某些专家被过度使用
5. **MLA + MoE 组合**: MLA 降低注意力内存，MoE 降低 FFN 计算量，两者结合实现高效大模型